<a href="https://colab.research.google.com/github/Najla-Nuricia/PemMes_244107020091_Najla/blob/main/JS06/JS06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
from google.colab import files
uploaded = files.upload()

Saving voice.csv to voice.csv


### Buatlah model SVM dengan menggunakan data voice.csv

In [3]:
df = pd.read_csv("voice.csv")

In [4]:
X = df.drop(columns=["label"])
y = df["label"]

In [5]:
y = y.map({"male": 0, "female": 1})

In [6]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

results = []

In [7]:
splits = {
    "70:30": 0.30,
    "80:20": 0.20
}

In [8]:
kernels = {
    "Linear": SVC(kernel="linear"),
    "Polynomial": SVC(kernel="poly"),
    "RBF": SVC(kernel="rbf")
}


In [9]:
for split_name, test_size in splits.items():

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    for kernel_name, model in kernels.items():

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        results.append({
            "Split": split_name,
            "Kernel": kernel_name,
            "Accuracy": accuracy
        })

results_df = pd.DataFrame(results)

results_df["Accuracy (%)"] = results_df["Accuracy"] * 100

results_df

,Split,Kernel,Accuracy,Accuracy (%)
0,70:30,Linear,0.970557,97.055731
1,70:30,Polynomial,0.949527,94.952681
2,70:30,RBF,0.981073,98.107256
3,80:20,Linear,0.968454,96.845426
4,80:20,Polynomial,0.949527,94.952681
5,80:20,RBF,0.979495,97.949527


### Gunakan data pada praktikum 5 untuk membuat model klasifikasi siang dan malam menggunakan SVM dengan kernel RBF menggunakan fitur histrogram. Gunakan rasio 80:20. Anda dapat bereksperimen dengan hyperparameter tunning dari kernel RBF. Catat performansi akurasinya!

In [13]:
import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [14]:
train_dir = "/content/drive/MyDrive/coolyeah /images/images/training"
test_dir = "/content/drive/MyDrive/coolyeah /images/images/test"

classes = {"day": 0,"night": 1}

In [15]:
def extract_histogram(image_path):
    image = cv2.imread(image_path)

    if image is None:
        return None

    # BGR -> RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Histogram channel R
    hist_r = cv2.calcHist(
        [image], [0], None, [256], [0, 256]
    )

    # Histogram channel G
    hist_g = cv2.calcHist(
        [image], [1], None, [256], [0, 256]
    )

    # Histogram channel B
    hist_b = cv2.calcHist(
        [image], [2], None, [256], [0, 256]
    )

    # Normalisasi
    hist_r = cv2.normalize(hist_r, hist_r).flatten()
    hist_g = cv2.normalize(hist_g, hist_g).flatten()
    hist_b = cv2.normalize(hist_b, hist_b).flatten()

    # Gabungkan menjadi satu feature vector
    features = np.concatenate([
        hist_r,
        hist_g,
        hist_b
    ])

    return features

In [16]:
#membaca semua data lalu di split 80: 20
X = []
y = []

def load_images_from_folder(base_dir):

    for class_name, label in classes.items():

        class_dir = os.path.join(base_dir, class_name)

        if not os.path.exists(class_dir):
            print(f"Folder tidak ditemukan: {class_dir}")
            continue

        for filename in os.listdir(class_dir):

            filepath = os.path.join(class_dir, filename)

            # Hanya proses file gambar
            if not filename.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            ):
                continue

            features = extract_histogram(filepath)

            if features is not None:
                X.append(features)
                y.append(label)


# Load training
load_images_from_folder(train_dir)

# Load test
load_images_from_folder(test_dir)

X = np.array(X)
y = np.array(y)

print("Jumlah gambar :", len(X))
print("Jumlah fitur  :", X.shape[1])
print("Shape X       :", X.shape)
print("Shape y       :", y.shape)

Jumlah gambar : 401
Jumlah fitur  : 768
Shape X       : (401, 768)
Shape y       : (401,)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data training :", len(X_train))
print("Data testing  :", len(X_test))

Data training : 320
Data testing  : 81


In [18]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
svm_rbf = SVC(
    kernel="rbf",
    C=1,
    gamma="scale"
)

svm_rbf.fit(X_train_scaled, y_train)

y_pred = svm_rbf.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy SVM RBF: {accuracy * 100:.2f}%")

Accuracy SVM RBF: 98.77%


In [20]:
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.001, 0.01, 0.1, 1]
}

grid_search = GridSearchCV(
    SVC(kernel="rbf"),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Parameter terbaik:")
print(grid_search.best_params_)

print(
    f"Best CV Accuracy: "
    f"{grid_search.best_score_ * 100:.2f}%"
)

Parameter terbaik:
{'C': 1, 'gamma': 'scale'}
Best CV Accuracy: 99.06%


In [21]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test_scaled)

final_accuracy = accuracy_score(
    y_test,
    y_pred_best
)

print(
    f"Final Test Accuracy: "
    f"{final_accuracy * 100:.2f}%"
)

Final Test Accuracy: 98.77%


In [22]:
print(
    classification_report(
        y_test,
        y_pred_best,
        target_names=["Day", "Night"]
    )
)

              precision    recall  f1-score   support

         Day       0.98      1.00      0.99        40
       Night       1.00      0.98      0.99        41

    accuracy                           0.99        81
   macro avg       0.99      0.99      0.99        81
weighted avg       0.99      0.99      0.99        81



In [23]:
cm = confusion_matrix(y_test, y_pred_best)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[40  0]
 [ 1 40]]


In [24]:
results = pd.DataFrame(grid_search.cv_results_)

results_table = results[
    [
        "param_C",
        "param_gamma",
        "mean_test_score",
        "std_test_score"
    ]
].sort_values(
    "mean_test_score",
    ascending=False
)

results_table["mean_test_score"] *= 100
results_table["std_test_score"] *= 100

results_table

,param_C,param_gamma,mean_test_score,std_test_score
11,10.0,0.001,99.0625,0.765466
10,10.0,scale,99.0625,0.765466
6,1.0,0.001,99.0625,0.765466
5,1.0,scale,99.0625,0.765466
16,100.0,0.001,99.0625,0.765466
15,100.0,scale,99.0625,0.765466
7,1.0,0.01,98.4375,1.397542
17,100.0,0.01,98.4375,1.397542
12,10.0,0.01,98.4375,1.397542
0,0.1,scale,95.0000,2.688227
